# Video Generation Overview

This notebook introduces the core concepts behind diffusion-based video generation, using the Wan 2.1 architecture as our reference and a miniature **NanoDiT** model as our hands-on learning tool.

## What is Video Generation?

Video generation is the task of synthesizing realistic video frames from a conditioning signal -- typically a text prompt. At first glance it may seem like a straightforward extension of image generation, but adding the **temporal dimension** introduces several major challenges:

| Aspect | Image Generation | Video Generation |
|--------|-----------------|------------------|
| Dimensionality | 2D (H x W) | 3D (T x H x W) |
| Consistency | Single frame | Must be coherent across many frames |
| Compute | Manageable | Grows linearly (or worse) with frame count |
| Motion | N/A | Must model physically plausible motion |

Modern video diffusion models like **Wan 2.1** tackle these challenges with three core components:

1. **Video VAE (Variational Autoencoder)** -- compresses raw pixel-space video into a compact *latent* representation. This reduces the spatial and temporal resolution, making diffusion tractable.

2. **Text Encoder** -- converts a natural-language prompt into a sequence of dense embeddings that the diffusion model can condition on. Wan uses a large T5-style encoder (4096-dim, ~4B params). Our nano model uses a simple embedding table.

3. **Diffusion Transformer (DiT)** -- the core generative model. It learns to iteratively *denoise* a random noise sample in latent space, guided by the text conditioning. The result is decoded by the VAE back into pixel space.

## Pipeline Flow

The full text-to-video generation pipeline works as follows:

```
Text Prompt
    |  
    v
[Text Encoder] --> text embeddings (B, seq_len, text_dim)
    |  
    v
Random Noise z_T ~ N(0, I)  in latent space (B, C_lat, T', H', W')
    |  
    v  (iterative denoising, N steps)
[DiT Model]  x  text embeddings  x  timestep  -->  predicted velocity
    |  
    v  (Euler step: z_{t-1} = z_t + v * delta_sigma)
Clean latent z_0  (B, C_lat, T', H', W')
    |  
    v
[VAE Decoder]  -->  video pixels  (B, 3, T, H, W)
```

During **training**, the pipeline runs in reverse: real videos are encoded to latents, noise is added at a random timestep, and the model learns to predict the velocity (noise - clean).

In [ ]:
# Architecture comparison: Wan 14B vs Wan 1.3B vs Nano
#
# This table shows how our Nano model preserves every architectural concept
# from the full Wan model, just at a much smaller scale.

header = f"{'Component':<20} {'Wan 14B':>10} {'Wan 1.3B':>10} {'Nano':>10}"
sep    = '-' * len(header)

rows = [
    ('DiT dim',           '5120',  '1536',  '128'),
    ('Attention heads',   '40',    '12',    '4'),
    ('Transformer layers','40',    '30',    '2'),
    ('FFN dim',           '13824', '8960',  '512'),
    ('Latent channels',   '16',    '16',    '4'),
    ('Text dim',          '4096',  '4096',  '64'),
    ('Freq dim',          '256',   '256',   '64'),
    ('Patch size',        '[1,2,2]','[1,2,2]','[1,2,2]'),
    ('~Parameters',       '~14B',  '~1.3B', '~1.5M'),
]

print(header)
print(sep)
for name, wan14, wan1, nano in rows:
    print(f"{name:<20} {wan14:>10} {wan1:>10} {nano:>10}")

## Why Latent Diffusion?

Running diffusion directly in pixel space is prohibitively expensive for video. Consider a modest 16-frame, 64x64 video:

- **Pixel space**: `B x 3 x 16 x 64 x 64` = 196,608 values per sample
- **Latent space**: `B x 4 x 4 x 16 x 16` = 4,096 values per sample

That is a **48x** compression ratio! The VAE learns to preserve perceptually important information while discarding redundancy (neighboring pixels are highly correlated, and consecutive frames share most of their content).

The diffusion model operates entirely in this compressed latent space, making both training and inference dramatically faster and more memory-efficient. The VAE decoder then upsamples the denoised latent back to full-resolution video.

This is the core idea behind *Latent Diffusion Models* (Rombach et al., 2022), adopted by Stable Diffusion, FLUX, and Wan alike.

In [ ]:
# Compute the compression ratio concretely

# Pixel-space video dimensions
B, C, T, H, W = 1, 3, 16, 64, 64
pixel_values = C * T * H * W

# Latent-space dimensions (after VAE encoding)
# Spatial factor = 4 (64 -> 16), Temporal factor = 4 (16 -> 4), Channels = 4
C_lat = 4
T_lat = T // 4   # = 4
H_lat = H // 4   # = 16
W_lat = W // 4   # = 16
latent_values = C_lat * T_lat * H_lat * W_lat

ratio = pixel_values / latent_values

print(f"Pixel space:  [{B}, {C}, {T}, {H}, {W}] = {pixel_values:,} values per sample")
print(f"Latent space: [{B}, {C_lat}, {T_lat}, {H_lat}, {W_lat}] = {latent_values:,} values per sample")
print(f"Compression ratio: {ratio:.0f}x")
print()
print("This means the DiT model only needs to denoise ~4K values")
print("instead of ~197K values per diffusion step!")

## Tutorial Roadmap

This tutorial series is organized into the following notebooks:

| # | Notebook | What you will learn |
|---|----------|---------------------|
| 1 | **01 -- Video Gen Overview** (this notebook) | High-level architecture, latent diffusion motivation, pipeline flow |
| 2 | **02 -- VAE & Latent Space** | How the VAE compresses video, encode/decode demo, latent visualization |
| 3 | **03 -- Building Blocks of the DiT** | Every component in detail: sinusoidal embeddings, 3D RoPE, RMSNorm, AdaIN modulation, self-attention, cross-attention, FFN, gating |

Each notebook is self-contained and runnable. The code uses our `nano_video_gen` package, which mirrors every architectural concept from Wan 2.1 at a scale small enough to run on a single CPU or a modest GPU.

Let's get started!